In [113]:
import pandas as pd
import numpy as np
from sklearn.ensemble import IsolationForest
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import StandardScaler
from scipy.stats import entropy
from scipy.signal import periodogram
import joblib
import anom_detection_isolation_forest as adif
import glob
import os
import random

In [ ]:
#pipeline configuration
MODEL_PATH = "iso_forest_nitrogen.pkl"
SCALER_PATH = "iso_scaler.pkl"
SPLIT_SIZE =  60 # Size of the rolling window in minutes

In [115]:
# Config
DATA_FOLDER = "../PW_0.5h"
TARGET = DATA_FOLDER
TRAIN_PATH = f"{DATA_FOLDER}/train.csv"
VAL_PATH = f"{DATA_FOLDER}/val.csv"
TEST_PATH = f"{DATA_FOLDER}/test.csv"

BENCHMARKS_FOLDER = "../Synthetic Sensor Fault/synthetic_validation_fault_outputs/benchmarks"

SENSORS = ["CMS air pressure", "Concentrazione ossigeno base", "Nitrogen pressure", "Oxygen over threshold"]

In [116]:
#TD: fine tune ranges, jump thresholds, and baseline stats based on domain knowledge
RANGES = {
    "CMS air pressure": (0, 9.5),
    "Concentrazione ossigeno base": (5, 500),
    "Nitrogen pressure": (0, 9),
    "Oxygen over threshold": (0, 1)
}

JUMP_THRESHOLDS = {
    "CMS air pressure": 5,
    "Concentrazione ossigeno base": 1,
    "Nitrogen pressure": 0.2,
    "Oxygen over threshold": 0.1
}

BASELINE_STATS = {
    "CMS air pressure": (1000, 10),
    "Concentrazione ossigeno base": (21, 1),
    "Nitrogen pressure": (2.0, 0.2),
    "Oxygen over threshold": (-0.045, 22.5)
}

In [117]:
#split data into RWs of 30 rows
def split_into_rws(df, rw_size):
    return [df.iloc[i:i + rw_size] for i in range(0, len(df), rw_size)]

In [118]:
# Load data
def load_data():
    train = pd.read_csv(TRAIN_PATH)

    #need to drop timestamp, PW_0.5h col
    train = train.drop(columns=['timestamp', 'PW_0.5h'])

    return train


In [119]:
#train
def train_isolation_forest(feature_matrix):
    """
    Train Isolation Forest on extracted features.
    """
    # Scale features
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(feature_matrix)

    # Train model
    iso = IsolationForest(
        contamination='auto'
    )
    iso.fit(X_scaled)

    # Save model + scaler
    joblib.dump(iso, MODEL_PATH)
    joblib.dump(scaler, SCALER_PATH)

    print("Isolation Forest training complete.")
    print(f"Model saved to: {MODEL_PATH}")
    print(f"Scaler saved to: {SCALER_PATH}")

    return iso, scaler

In [120]:
#Load model for runtime use
def load_iso_model():
    iso = joblib.load(MODEL_PATH)
    scaler = joblib.load(SCALER_PATH)
    return iso, scaler

In [121]:
#compute trust score for a single RW
def compute_trust_score(rw, ranges, jump_thresholds, baseline_stats):
    """
    Returns trust score and anomaly flag for a single RW.
    """
    iso, scaler = load_iso_model()

    feats = adif.extract_features(rw, ranges, jump_thresholds, baseline_stats)
    X_scaled = scaler.transform(feats)

    #Trust score: higher = more normal, lower = more anomalous [0, 1]
    trust_score = iso.score_samples(X_scaled)[0]*-1 
    # Anomaly flag (0 = normal, 1 = anomaly)
    anomaly = iso.predict(X_scaled)[0]== -1

    return trust_score, anomaly

In [122]:
#batch scoring for multiple RWs
def score_many_rws(rw_list, ranges, jump_thresholds, baseline_stats):
    iso, scaler = load_iso_model()

    results = []
    for rw in rw_list:
        window_start = rw['timestamp'].iloc[0]
        window_end = rw['timestamp'].iloc[-1]
        sensor_readings = rw.drop(columns=['timestamp'])
        # extract_features summarizes every reading in this RW into one feature row.
        feats = adif.extract_features(sensor_readings, ranges, jump_thresholds, baseline_stats)
        X_scaled = scaler.transform(feats)

        trust_score = -iso.score_samples(X_scaled).item()
        # .item() extracts the one prediction for the whole RW's feature row.
        anomaly = int(iso.predict(X_scaled).item())


        
        results.append({
        "timestamp": window_start,
        "end_timestamp": window_end,
        "trust_score": trust_score,
        "anomaly": anomaly,
    })

    return pd.DataFrame(results)

In [ ]:
validation_files = sorted(glob.glob(os.path.join(BENCHMARKS_FOLDER, "*.csv")))
if not validation_files:
    raise FileNotFoundError(f"No validation CSV files found in {BENCHMARKS_FOLDER}")
validation_files = random.sample(validation_files, k=min(5, len(validation_files)))

required_columns = ["timestamp", *SENSORS]
report_path = os.path.join(os.path.dirname(BENCHMARKS_FOLDER), "validation_anomaly_summary.txt")
report_lines = []
scores_by_dataset = {}

for filename in validation_files:
    dataset_name = os.path.basename(filename)
    val_df = pd.read_csv(filename)
    missing_columns = [column for column in required_columns if column not in val_df.columns]
    if missing_columns:
        raise ValueError(f"{dataset_name} is missing required columns: {missing_columns}")

    val_features = val_df[required_columns]
    val_split_rws = split_into_rws(val_features, SPLIT_SIZE)  # Assuming SPLIT_SIZE is defined somewhere in your code
    scores = score_many_rws(val_split_rws, RANGES, JUMP_THRESHOLDS, BASELINE_STATS)
    scores_by_dataset[dataset_name] = scores

    anomalies = scores[scores["anomaly"] == -1]
    total_windows = len(scores)
    anomaly_count = len(anomalies)
    anomaly_rate = (anomaly_count / total_windows * 100) if total_windows else 0

    report_lines.extend([
        f"Dataset: {dataset_name}",
        "=" * (len(dataset_name) + 9),
        f"Total reading windows: {total_windows}",
        f"Anomalous windows: {anomaly_count}",
        f"Normal windows: {total_windows - anomaly_count}",
        f"Anomaly rate: {anomaly_rate:.2f}%",
    ])

    if anomalies.empty:
        report_lines.append("No anomalies found.")
    else:
        report_lines.extend([
            f"First anomaly: {anomalies['timestamp'].iloc[0]}",
            f"Last anomaly: {anomalies['timestamp'].iloc[-1]}",
            f"Mean anomaly trust score: {anomalies['trust_score'].mean():.4f}",
            f"Highest anomaly trust score: {anomalies['trust_score'].max():.4f}",
            "Detected anomaly windows:",
            anomalies[["timestamp", "end_timestamp", "trust_score"]].to_string(index=False),
        ])

    report_lines.append("")

with open(report_path, "w", encoding="utf-8") as report_file:
    report_file.write("\n".join(report_lines))

print(f"Processed {len(validation_files)} validation datasets.")
print(f"Consolidated anomaly report saved to: {os.path.abspath(report_path)}")



Processed 5 validation datasets.
Consolidated anomaly report saved to: c:\Users\gavri\OneDrive\Documents\IBMZ datathon\IBM_Z_Datathon\Synthetic Sensor Fault\synthetic_validation_fault_outputs\validation_anomaly_summary.txt


In [124]:
print(f"Consolidated anomaly report: {os.path.abspath(report_path)}")

Consolidated anomaly report: c:\Users\gavri\OneDrive\Documents\IBMZ datathon\IBM_Z_Datathon\Synthetic Sensor Fault\synthetic_validation_fault_outputs\validation_anomaly_summary.txt


In [ ]:
# MAIN WORKFLOW

def main():
    

    train = load_data()


    #split data into RWs of 30 minutes
    train_rw = split_into_rws(train, SPLIT_SIZE)  # Assuming SPLIT_SIZE is defined somewhere in your code

    feature_matrix = adif.build_feature_matrix(train_rw, RANGES, JUMP_THRESHOLDS, BASELINE_STATS)
    iso, scaler = train_isolation_forest(feature_matrix)

    

    return {
        "model": iso
    }   

if __name__ == "__main__":
    artifacts = main()
    
    
    

Isolation Forest training complete.
Model saved to: iso_forest_nitrogen.pkl
Scaler saved to: iso_scaler.pkl
